### Step 1 — Verify the encoded training data

Load the 1,000-image MNIST training subset. Check its shape, class balance, encoding settings, packed-word format, and file hash before using it for hardware training.

In [1]:
from pathlib import Path
import hashlib
import numpy as np

train_file = Path(
    "/home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train1000_words.npz"
)
assert train_file.is_file(), f"Missing file: {train_file}"

with np.load(train_file, allow_pickle=False) as data:
    train_words = data["words"]
    train_labels = data["labels"]
    train_indices = data["indices"]
    train_spike_counts = data["spike_counts"]
    train_num_steps = int(data["num_steps"])
    train_gain = float(data["gain"])
    train_seed = int(data["seed"])
    train_split = str(data["split"].item())
    train_bitorder = str(data["bitorder"].item())

class_counts = np.bincount(
    train_labels.astype(np.int32), minlength=10
)
file_hash = hashlib.sha256(train_file.read_bytes()).hexdigest()

assert train_words.shape == (1000, 100, 25)
assert train_words.dtype == np.uint32
assert train_labels.shape == (1000,)
assert train_indices.shape == (1000,)
assert np.unique(train_indices).size == 1000
assert train_spike_counts.shape == (1000,)
assert np.array_equal(class_counts, np.full(10, 100))
assert np.all((train_words[:, :, 24] & 0xFFFF0000) == 0)
assert train_num_steps == 100
assert np.isclose(train_gain, 0.1)
assert train_split == "train"
assert train_bitorder == "little"

print("File:", train_file)
print("File size:", train_file.stat().st_size, "bytes")
print("Words shape:", train_words.shape)
print("Class counts:", class_counts.tolist())
print("Total input spikes:", int(train_spike_counts.sum()))
print("Encoding seed:", train_seed)
print("SHA256:", file_hash)
print("Training data check passed")

File: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train1000_words.npz
File size: 1342430 bytes
Words shape: (1000, 100, 25)
Class counts: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Total input spikes: 1020741
Encoding seed: 2026
SHA256: 2c8d4acdd484a6106102afb2fc731dd1515e98efe32451f5265c54517e141d7a
Training data check passed


### Step 2 — Load the overlay and configure training parameters

Program the FPGA with fresh initial weights. Set the calcium leak interval to 10 steps, disable bistability events, and set the teacher current to 0.25 in Q9 format. Confirm the register values before sending training data.

In [2]:
from pynq import Overlay, MMIO

bit_file = train_file.parent / "snn_sdsp.bit"
hwh_file = train_file.parent / "snn_sdsp.hwh"

assert bit_file.is_file(), f"Missing bitstream: {bit_file}"
assert hwh_file.is_file(), f"Missing metadata: {hwh_file}"

print("Programming FPGA...", flush=True)

overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

ca_interval_steps = 10
bist_interval_steps = 0
teacher_current_q9 = 128

snn.write(0x10, ca_interval_steps)
snn.write(0x14, bist_interval_steps)
snn.write(0x18, teacher_current_q9)

control = snn.read(0x00)
status = snn.read(0x04)
n_cycles = snn.read(0x08)
ca_readback = snn.read(0x10)
bist_readback = snn.read(0x14)
teacher_readback = snn.read(0x18)
dma_status = dma.mmio.read(0x04)

print("Overlay loaded:", overlay.is_loaded())
print(f"CONTROL: 0x{control:08X}")
print(f"SNN status: 0x{status:08X}")
print("N_CYCLES:", n_cycles)
print("CA interval:", ca_readback)
print("BIST interval:", bist_readback)
print("Teacher current Q9:", teacher_readback)
print(f"DMA status: 0x{dma_status:08X}")
print("DMA running:", dma.sendchannel.running)

assert overlay.is_loaded()
assert (control & 0x44) == 0
assert status & 0x01
assert n_cycles == 100
assert ca_readback == 10
assert bist_readback == 0
assert teacher_readback == 128
assert dma.sendchannel.running
assert (dma_status & 0x70) == 0

print("Training hardware configuration passed")

Programming FPGA...


Overlay loaded: True
CONTROL: 0x00000000
SNN status: 0x00000001
N_CYCLES: 100
CA interval: 10
BIST interval: 0
Teacher current Q9: 128
DMA status: 0x00000000
DMA running: True
Training hardware configuration passed


### Step 3 — Load fixed evaluation samples

Load the ten previously encoded MNIST test images. These samples will remain separate from the 1,000 training images and provide a fixed before-and-after comparison.

In [3]:
eval_file = train_file.parent / "mnist_test10_spikes.npz"
assert eval_file.is_file(), f"Missing evaluation file: {eval_file}"

with np.load(eval_file, allow_pickle=False) as data:
    eval_words = data["words"]
    eval_labels = data["labels"]
    eval_indices = data["indices"]
    eval_num_steps = int(data["num_steps"])
    eval_gain = float(data["gain"])

assert eval_words.shape == (10, 100, 25)
assert eval_words.dtype == np.uint32
assert eval_labels.shape == (10,)
assert eval_num_steps == train_num_steps
assert np.isclose(eval_gain, train_gain)
assert np.all((eval_words[:, :, 24] & 0xFFFF0000) == 0)

print("Evaluation file:", eval_file)
print("Words shape:", eval_words.shape)
print("Labels:", eval_labels.tolist())
print("Time steps:", eval_num_steps)
print("Encoding gain:", eval_gain)
print("Fixed evaluation data check passed")

Evaluation file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_test10_spikes.npz
Words shape: (10, 100, 25)
Labels: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
Time steps: 100
Encoding gain: 0.1
Fixed evaluation data check passed


### Step 4 — Verify one inference transfer

Define a reusable single-frame transfer function with DMA and SNN timeouts. Run one evaluation image with learning and the teacher signal disabled before recording the full baseline.

In [4]:
import time
from pynq import allocate


def run_frame(frame, label=None, learn=False, teacher=False, timeout=10.0):
    frame = np.asarray(frame, dtype=np.uint32)
    assert frame.shape == (100, 25)
    assert not teacher or (learn and label is not None)

    tx = allocate(shape=(2500,), dtype=np.uint32)

    try:
        tx[:] = frame.reshape(-1)
        tx.flush()

        snn.write(0x00, 0x02)
        time.sleep(0.01)

        snn.write(0x0C, int(label) if teacher else 0)
        snn.write(0x10, ca_interval_steps)
        snn.write(0x14, bist_interval_steps)
        snn.write(0x18, teacher_current_q9)

        control = 0x01
        if learn:
            control |= 0x04
        if teacher:
            control |= 0x40

        start_time = time.monotonic()
        snn.write(0x00, control)
        dma.sendchannel.transfer(tx)

        deadline = time.monotonic() + timeout
        while not dma.sendchannel.idle:
            dma_status = dma.mmio.read(0x04)
            if dma_status & 0x70:
                raise RuntimeError(
                    f"DMA error: 0x{dma_status:08X}"
                )
            if time.monotonic() > deadline:
                raise TimeoutError("DMA transfer timeout")
            time.sleep(0.001)

        dma.sendchannel.wait()

        deadline = time.monotonic() + timeout
        while not (snn.read(0x04) & 0x04):
            if time.monotonic() > deadline:
                raise TimeoutError("SNN completion timeout")
            time.sleep(0.001)

        result = {
            "status": snn.read(0x04),
            "accepted": snn.read(0x50),
            "spike_counts": [
                snn.read(0x24 + 4 * index) for index in range(10)
            ],
            "result_class": snn.read(0x20),
            "control": snn.read(0x00),
            "dma_status": dma.mmio.read(0x04),
            "elapsed_seconds": time.monotonic() - start_time,
        }

        assert result["accepted"] == 100
        assert (result["status"] & 0x18) == 0
        assert (result["dma_status"] & 0x70) == 0
        assert (result["control"] & 0x44) == (control & 0x44)

        return result

    finally:
        tx.close()


eval_index = 3
single_result = run_frame(eval_words[eval_index])

print("Evaluation index:", eval_index)
print("True label:", int(eval_labels[eval_index]))
print(f"DMA status: 0x{single_result['dma_status']:08X}")
print(f"SNN status: 0x{single_result['status']:08X}")
print("Accepted steps:", single_result["accepted"])
print("Output spike counts:", single_result["spike_counts"])
print("Result class:", single_result["result_class"])
print("Elapsed seconds:", round(single_result["elapsed_seconds"], 4))
print("Single-frame inference passed")

Evaluation index: 3
True label: 0
DMA status: 0x00001002
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [6, 5, 6, 6, 5, 6, 6, 6, 5, 6]
Result class: 0
Elapsed seconds: 0.0023
Single-frame inference passed


### Step 5 — Record the pre-training baseline

Run inference on all ten fixed evaluation images with learning and the teacher signal disabled. Keep their spike counts in memory for comparison after the training pilot.

In [6]:
baseline_results = []
baseline_start = time.monotonic()

for index in range(len(eval_labels)):
    result = run_frame(eval_words[index])
    baseline_results.append(result)

    print(
        f"Index={index} "
        f"Label={int(eval_labels[index])} "
        f"Result={result['result_class']} "
        f"Counts={result['spike_counts']}"
    )

baseline_wall_seconds = time.monotonic() - baseline_start

baseline_counts = np.asarray(
    [result["spike_counts"] for result in baseline_results],
    dtype=np.uint32,
)
baseline_classes = np.asarray(
    [result["result_class"] for result in baseline_results],
    dtype=np.uint32,
)

silent_samples = int(np.count_nonzero(
    np.all(baseline_counts == 0, axis=1)
))

assert baseline_counts.shape == (10, 10)
assert baseline_classes.shape == (10,)

print("Completed evaluation images:", len(baseline_results))
print("Silent samples:", silent_samples)
print("Total wall time:", round(baseline_wall_seconds, 2), "seconds")
print("Pre-training baseline recorded")

Index=0 Label=7 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=1 Label=2 Result=2 Counts=[3, 3, 4, 3, 3, 3, 3, 3, 3, 3]
Index=2 Label=1 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=3 Label=0 Result=0 Counts=[6, 5, 6, 6, 5, 6, 6, 6, 5, 6]
Index=4 Label=4 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=5 Label=1 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=6 Label=4 Result=0 Counts=[1, 0, 0, 0, 0, 0, 1, 1, 0, 0]
Index=7 Label=9 Result=0 Counts=[2, 1, 1, 0, 1, 1, 1, 1, 1, 0]
Index=8 Label=5 Result=0 Counts=[4, 4, 4, 3, 4, 4, 4, 4, 4, 4]
Index=9 Label=9 Result=0 Counts=[4, 3, 3, 3, 4, 4, 4, 4, 4, 4]
Completed evaluation images: 10
Silent samples: 4
Total wall time: 0.16 seconds
Pre-training baseline recorded


### Step 6 — Train the first image

Run one MNIST training image with learning and the label-specific teacher signal enabled. Confirm completion, register settings, and the target neuron's response. Keep the overlay loaded afterward.

In [7]:
train_position = 0
train_label = int(train_labels[train_position])

first_train_result = run_frame(
    train_words[train_position],
    label=train_label,
    learn=True,
    teacher=True,
)
trained_count = 1

print("Training position:", train_position)
print("MNIST train index:", int(train_indices[train_position]))
print("Teacher label:", train_label)
print("Input spikes:", int(train_spike_counts[train_position]))
print(f"CONTROL: 0x{first_train_result['control']:08X}")
print(f"DMA status: 0x{first_train_result['dma_status']:08X}")
print(f"SNN status: 0x{first_train_result['status']:08X}")
print("Accepted steps:", first_train_result["accepted"])
print("Output spike counts:", first_train_result["spike_counts"])
print(
    "Target neuron spikes:",
    first_train_result["spike_counts"][train_label],
)
print("CA interval:", snn.read(0x10))
print("BIST interval:", snn.read(0x14))

assert first_train_result["control"] == 0x44
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0

print("First supervised training frame passed")

Training position: 0
MNIST train index: 28065
Teacher label: 8
Input spikes: 1689
CONTROL: 0x00000044
DMA status: 0x00001002
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [4, 5, 5, 5, 4, 5, 5, 4, 35, 5]
Target neuron spikes: 35
CA interval: 10
BIST interval: 0
First supervised training frame passed


### Step 7 — Train twenty images in total

Continue from training position 1 through position 19 without reloading the bitstream. Each image uses its own true label as the teacher target. Track the number of completed images so a failed run is not accidentally repeated.

In [8]:
assert trained_count == 1, (
    f"Expected one completed image, found {trained_count}"
)

pilot_training_results = [first_train_result]
pilot_start = time.monotonic()

for position in range(trained_count, 20):
    label = int(train_labels[position])

    result = run_frame(
        train_words[position],
        label=label,
        learn=True,
        teacher=True,
    )

    pilot_training_results.append(result)
    trained_count = position + 1

    print(
        f"Position={position} "
        f"Label={label} "
        f"InputSpikes={int(train_spike_counts[position])} "
        f"TargetSpikes={result['spike_counts'][label]} "
        f"Accepted={result['accepted']}"
    )

pilot_wall_seconds = time.monotonic() - pilot_start

assert trained_count == 20
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0

print("Completed training images:", trained_count)
print("Additional training wall time:",
      round(pilot_wall_seconds, 2), "seconds")
print("Twenty-image training check passed")

Position=1 Label=0 InputSpikes=1133 TargetSpikes=29 Accepted=100
Position=2 Label=4 InputSpikes=1444 TargetSpikes=31 Accepted=100
Position=3 Label=3 InputSpikes=927 TargetSpikes=26 Accepted=100
Position=4 Label=7 InputSpikes=682 TargetSpikes=25 Accepted=100
Position=5 Label=6 InputSpikes=853 TargetSpikes=26 Accepted=100
Position=6 Label=3 InputSpikes=711 TargetSpikes=26 Accepted=100
Position=7 Label=5 InputSpikes=1357 TargetSpikes=30 Accepted=100
Position=8 Label=2 InputSpikes=1189 TargetSpikes=28 Accepted=100
Position=9 Label=2 InputSpikes=1678 TargetSpikes=31 Accepted=100
Position=10 Label=1 InputSpikes=562 TargetSpikes=24 Accepted=100
Position=11 Label=2 InputSpikes=846 TargetSpikes=27 Accepted=100
Position=12 Label=3 InputSpikes=1225 TargetSpikes=28 Accepted=100
Position=13 Label=5 InputSpikes=1044 TargetSpikes=27 Accepted=100
Position=14 Label=7 InputSpikes=1166 TargetSpikes=27 Accepted=100
Position=15 Label=3 InputSpikes=1411 TargetSpikes=31 Accepted=100
Position=16 Label=5 Input

### Step 8 — Complete the 1,000-image training pilot

Continue from image 20 through image 999 without reloading the bitstream. Report progress every 100 images. Evaluate the fixed test samples only after all 1,000 training images are complete.

In [9]:
assert trained_count == 20, (
    f"Expected 20 completed images, found {trained_count}"
)

training_target_counts = np.empty(len(train_labels), dtype=np.uint16)

for position in range(trained_count):
    label = int(train_labels[position])
    training_target_counts[position] = (
        pilot_training_results[position]["spike_counts"][label]
    )

remaining_start = time.monotonic()

for position in range(trained_count, len(train_labels)):
    label = int(train_labels[position])

    result = run_frame(
        train_words[position],
        label=label,
        learn=True,
        teacher=True,
    )

    training_target_counts[position] = result["spike_counts"][label]
    trained_count = position + 1

    if trained_count % 100 == 0:
        print(
            f"Completed {trained_count}/{len(train_labels)} images",
            flush=True,
        )

remaining_wall_seconds = time.monotonic() - remaining_start

assert trained_count == len(train_labels)
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0

print("Completed training images:", trained_count)
print("Remaining training wall time:",
      round(remaining_wall_seconds, 2), "seconds")
print("Mean target spikes:",
      round(float(training_target_counts.mean()), 2))
print("Training pilot passed")

Completed 100/1000 images
Completed 200/1000 images
Completed 300/1000 images
Completed 400/1000 images
Completed 500/1000 images
Completed 600/1000 images
Completed 700/1000 images
Completed 800/1000 images
Completed 900/1000 images
Completed 1000/1000 images
Completed training images: 1000
Remaining training wall time: 14.38 seconds
Mean target spikes: 28.25
Training pilot passed


### Step 9 — Evaluate after the 1,000-image pilot

Run the fixed ten-image evaluation set with learning and the teacher signal disabled. Compare output spike counts with the pre-training baseline without reloading the bitstream.

In [10]:
assert trained_count == 1000

after1000_results = []

for index in range(len(eval_labels)):
    result = run_frame(eval_words[index])
    after1000_results.append(result)

    print(
        f"Index={index} "
        f"Label={int(eval_labels[index])} "
        f"Before={baseline_counts[index].tolist()} "
        f"After={result['spike_counts']}"
    )

after1000_counts = np.asarray(
    [result["spike_counts"] for result in after1000_results],
    dtype=np.uint32,
)
after1000_classes = np.asarray(
    [result["result_class"] for result in after1000_results],
    dtype=np.uint32,
)

changed_indices = np.flatnonzero(
    np.any(after1000_counts != baseline_counts, axis=1)
)
silent_after1000 = int(np.count_nonzero(
    np.all(after1000_counts == 0, axis=1)
))

assert after1000_counts.shape == baseline_counts.shape
assert (snn.read(0x00) & 0x44) == 0

print("Changed evaluation samples:", changed_indices.tolist())
print("Silent samples before:", silent_samples)
print("Silent samples after:", silent_after1000)
print("Total output spikes before:", int(baseline_counts.sum()))
print("Total output spikes after:", int(after1000_counts.sum()))
print("Post-training evaluation passed")

Index=0 Label=7 Before=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=1 Label=2 Before=[3, 3, 4, 3, 3, 3, 3, 3, 3, 3] After=[0, 0, 3, 1, 0, 0, 0, 0, 0, 0]
Index=2 Label=1 Before=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=3 Label=0 Before=[6, 5, 6, 6, 5, 6, 6, 6, 5, 6] After=[3, 0, 3, 0, 0, 1, 1, 0, 1, 1]
Index=4 Label=4 Before=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=5 Label=1 Before=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=6 Label=4 Before=[1, 0, 0, 0, 0, 0, 1, 1, 0, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=7 Label=9 Before=[2, 1, 1, 0, 1, 1, 1, 1, 1, 0] After=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index=8 Label=5 Before=[4, 4, 4, 3, 4, 4, 4, 4, 4, 4] After=[0, 0, 2, 0, 0, 0, 0, 0, 0, 1]
Index=9 Label=9 Before=[4, 3, 3, 3, 4, 4, 4, 4, 4, 4] After=[0, 0, 0, 0, 0, 0, 0, 1, 1, 1]
Changed evaluation samples: [1, 3, 6, 7, 8, 9]
Silent samples before: 4
Silent samples aft

### Step 10 — Check the fixed evaluation dataset

Load and validate the pre-encoded MNIST test images. This cell does not program the FPGA or change its learned weights.

In [5]:
from pathlib import Path
import hashlib
import numpy as np

fixed_eval_file = Path(
    "/home/xilinx/jupyter_notebooks/snn_sdsp/mnist_test1000_fixed_words.npz"
)
assert fixed_eval_file.is_file()

with np.load(fixed_eval_file, allow_pickle=False) as data:
    fixed_eval_words = data["words"]
    fixed_eval_labels = data["labels"]
    fixed_eval_indices = data["indices"]
    fixed_eval_spike_counts = data["spike_counts"]
    fixed_eval_steps = int(data["num_steps"].item())
    fixed_eval_gain = float(data["gain"].item())
    fixed_eval_split = str(data["split"].item())
    fixed_eval_bitorder = str(data["bitorder"].item())

class_counts = np.bincount(
    fixed_eval_labels.astype(np.int32), minlength=10
)
fixed_eval_sha256 = hashlib.sha256(
    fixed_eval_file.read_bytes()
).hexdigest()

assert fixed_eval_words.shape == (1000, 100, 25)
assert fixed_eval_words.dtype == np.uint32
assert fixed_eval_labels.shape == (1000,)
assert fixed_eval_indices.shape == (1000,)
assert np.array_equal(class_counts, np.full(10, 100))
assert len(np.unique(fixed_eval_indices)) == 1000
assert np.all(fixed_eval_indices >= 10)
assert fixed_eval_steps == 100
assert np.isclose(fixed_eval_gain, 0.1)
assert fixed_eval_split == "test"
assert fixed_eval_bitorder == "little"

print("File:", fixed_eval_file)
print("File size:", fixed_eval_file.stat().st_size, "bytes")
print("Words shape:", fixed_eval_words.shape)
print("Class counts:", class_counts.tolist())
print("Total input spikes:", int(fixed_eval_spike_counts.sum()))
print("SHA256:", fixed_eval_sha256)
print("Fixed evaluation data check passed")

File: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_test1000_fixed_words.npz
File size: 1372377 bytes
Words shape: (1000, 100, 25)
Class counts: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Total input spikes: 1052896
SHA256: 2ade2e125d2d14257b72cbf7957de0d32c5e0879007bacce561242318f3d7e5e
Fixed evaluation data check passed


### Step 11 — Evaluate the fixed test set before training

Run inference on the same 1,000 encoded test images that will be used after training. Keep learning and teacher input disabled. Store output spike counts in notebook memory for the before-and-after comparison.

In [6]:
import time
import numpy as np

baseline1000_counts = np.empty((1000, 10), dtype=np.int32)

start_time = time.perf_counter()

for index in range(1000):
    result = run_frame(
        fixed_eval_words[index],
        learn=False,
        teacher=False,
    )

    counts = np.asarray(result["spike_counts"], dtype=np.int32)
    assert counts.shape == (10,)
    assert result["accepted"] == 100

    baseline1000_counts[index] = counts

    if (index + 1) % 100 == 0:
        print(f"Evaluated {index + 1}/1000 images", flush=True)

baseline1000_silent = baseline1000_counts.sum(axis=1) == 0
baseline1000_winners = baseline1000_counts.argmax(axis=1)
baseline1000_correct = (
    ~baseline1000_silent
    & (baseline1000_winners == fixed_eval_labels)
)

strict_accuracy = float(baseline1000_correct.mean())
argmax_accuracy = float(
    (baseline1000_winners == fixed_eval_labels).mean()
)
active_count = int((~baseline1000_silent).sum())
active_accuracy = (
    float(baseline1000_correct.sum() / active_count)
    if active_count else 0.0
)

print("Completed images:", len(baseline1000_counts))
print("Strict accuracy (silent = wrong):", round(strict_accuracy, 4))
print("Argmax accuracy (silent selects class 0):", round(argmax_accuracy, 4))
print("Accuracy among active images:", round(active_accuracy, 4))
print("Silent images:", int(baseline1000_silent.sum()))
print("Total output spikes:", int(baseline1000_counts.sum()))
print("Elapsed seconds:", round(time.perf_counter() - start_time, 2))
print("Pre-training evaluation completed")

Evaluated 100/1000 images
Evaluated 200/1000 images
Evaluated 300/1000 images
Evaluated 400/1000 images
Evaluated 500/1000 images
Evaluated 600/1000 images
Evaluated 700/1000 images
Evaluated 800/1000 images
Evaluated 900/1000 images
Evaluated 1000/1000 images
Completed images: 1000
Strict accuracy (silent = wrong): 0.059
Argmax accuracy (silent selects class 0): 0.063
Accuracy among active images: 0.0761
Silent images: 225
Total output spikes: 26653
Elapsed seconds: 14.82
Pre-training evaluation completed


### Step 12 — Train on the fixed 1,000-image training set

Present each saved training frame once, in its stored order, with learning and the correct teacher label enabled. Keep the current FPGA configuration and the pre-training evaluation results.

In [7]:
import time
import numpy as np

assert baseline1000_counts.shape == (1000, 10)
assert train_words.shape == (1000, 100, 25)
assert train_labels.shape == (1000,)
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0
assert snn.read(0x18) == 128

if "training1000_started" in globals():
    raise RuntimeError(
        "Training was already started in this kernel. Do not run this cell twice."
    )

training1000_started = True
training1000_completed = 0
training1000_target_counts = np.empty(1000, dtype=np.int32)
training_start = time.perf_counter()

for position in range(1000):
    label = int(train_labels[position])
    result = run_frame(
        train_words[position],
        label=label,
        learn=True,
        teacher=True,
    )

    counts = np.asarray(result["spike_counts"], dtype=np.int32)
    assert counts.shape == (10,)
    assert result["accepted"] == 100
    assert (result["control"] & 0x44) == 0x44

    training1000_target_counts[position] = counts[label]
    training1000_completed = position + 1

    if training1000_completed % 100 == 0:
        print(
            f"Trained {training1000_completed}/1000 images",
            flush=True,
        )

training1000_complete = True

print("Completed training images:", training1000_completed)
print(
    "Mean target-neuron spikes during training:",
    round(float(training1000_target_counts.mean()), 2),
)
print(
    "Training wall time (seconds):",
    round(time.perf_counter() - training_start, 2),
)
print("Training completed")

Trained 100/1000 images
Trained 200/1000 images
Trained 300/1000 images
Trained 400/1000 images
Trained 500/1000 images
Trained 600/1000 images
Trained 700/1000 images
Trained 800/1000 images
Trained 900/1000 images
Trained 1000/1000 images
Completed training images: 1000
Mean target-neuron spikes during training: 28.25
Training wall time (seconds): 14.8
Training completed


### Step 13 — Evaluate the fixed test set after training

Run inference on exactly the same 1,000 encoded test frames used before training. Disable learning and teacher input, then compare accuracy, silent images, and spike counts with the saved baseline.

In [8]:
import time
import numpy as np

assert training1000_complete
assert training1000_completed == 1000
assert baseline1000_counts.shape == (1000, 10)
assert fixed_eval_words.shape == (1000, 100, 25)

after1000_counts = np.empty((1000, 10), dtype=np.int32)
start_time = time.perf_counter()

for index in range(1000):
    result = run_frame(
        fixed_eval_words[index],
        learn=False,
        teacher=False,
    )

    counts = np.asarray(result["spike_counts"], dtype=np.int32)
    assert counts.shape == (10,)
    assert result["accepted"] == 100

    after1000_counts[index] = counts

    if (index + 1) % 100 == 0:
        print(f"Evaluated {index + 1}/1000 images", flush=True)

after1000_silent = after1000_counts.sum(axis=1) == 0
after1000_winners = after1000_counts.argmax(axis=1)
after1000_correct = (
    ~after1000_silent
    & (after1000_winners == fixed_eval_labels)
)

before_strict = float(baseline1000_correct.mean())
after_strict = float(after1000_correct.mean())
after_argmax = float(
    (after1000_winners == fixed_eval_labels).mean()
)
after_active_count = int((~after1000_silent).sum())
after_active_accuracy = (
    float(after1000_correct.sum() / after_active_count)
    if after_active_count else 0.0
)

print("Completed evaluation images:", len(after1000_counts))
print(
    "Strict accuracy before/after:",
    round(before_strict, 4),
    round(after_strict, 4),
)
print(
    "Argmax accuracy after (silent selects class 0):",
    round(after_argmax, 4),
)
print(
    "Accuracy among active images after:",
    round(after_active_accuracy, 4),
)
print(
    "Silent images before/after:",
    int(baseline1000_silent.sum()),
    int(after1000_silent.sum()),
)
print(
    "Total output spikes before/after:",
    int(baseline1000_counts.sum()),
    int(after1000_counts.sum()),
)
print(
    "Images with changed spike counts:",
    int(np.any(after1000_counts != baseline1000_counts, axis=1).sum()),
)

for digit in range(10):
    selected = fixed_eval_labels == digit
    print(
        f"Digit {digit}: "
        f"correct before={int(baseline1000_correct[selected].sum())}/100, "
        f"after={int(after1000_correct[selected].sum())}/100; "
        f"silent before={int(baseline1000_silent[selected].sum())}/100, "
        f"after={int(after1000_silent[selected].sum())}/100"
    )

print("Elapsed seconds:", round(time.perf_counter() - start_time, 2))
print("Post-training evaluation completed")

Evaluated 100/1000 images
Evaluated 200/1000 images
Evaluated 300/1000 images
Evaluated 400/1000 images
Evaluated 500/1000 images
Evaluated 600/1000 images
Evaluated 700/1000 images
Evaluated 800/1000 images
Evaluated 900/1000 images
Evaluated 1000/1000 images
Completed evaluation images: 1000
Strict accuracy before/after: 0.059 0.346
Argmax accuracy after (silent selects class 0): 0.367
Accuracy among active images after: 0.5545
Silent images before/after: 225 376
Total output spikes before/after: 26653 4690
Images with changed spike counts: 791
Digit 0: correct before=48/100, after=50/100; silent before=4/100, after=21/100
Digit 1: correct before=0/100, after=19/100; silent before=83/100, after=78/100
Digit 2: correct before=0/100, after=92/100; silent before=4/100, after=7/100
Digit 3: correct before=0/100, after=32/100; silent before=20/100, after=29/100
Digit 4: correct before=2/100, after=8/100; silent before=26/100, after=64/100
Digit 5: correct before=2/100, after=11/100; silen